In [ ]:
import mg5qs_imports as qs
from pathlib import Path
import os
import numpy as np

### Changing parameters across two dimensions

This example demonstrates how to change paramaters across two (or more) dimensions. 

This example includes:
- setting multiple parameters independently
- generating and showering events
- reassembling results in a coherent order
- plotting results across two dimensions

### Choose a process that generates $\tau$ particles

Namely: <code>generate p p > ta- vt~</code>, which primarially goes through W production with a resonance at the mass of the W boson. 

In [ ]:
INPUT_PATH = Path.cwd()/'input' # madgraph cards
# note that for this example, we move to a process which directly generates taus
qs.edit_card(INPUT_PATH, card_name='proc_card_example4.dat')

In [ ]:
output_name, FRAMEWORK_PATH = qs.run_MG5(INPUT_PATH, proc_card_name='proc_card_example4.dat')

### Retrieve standard values

The ParamCard class constructs a Python object which is an abstract of MadGraph's param_card.dat. These files hold run information such as particle masses, decay rates, etc. These parameters are accsessed at the time of LHE genoration.

First, let's construct a ParamCard object and take a look at some of the basic functionality.

In [ ]:
card = qs.ParamCard(FRAMEWORK_PATH / 'Cards' / 'param_card.dat') # construct card
card # display card abstract

A ParamCard is constructed from a number of pandas dataframes, which store the parameters directly. We can acsess these dataframes using ParamCard.dfs().

In [ ]:
card.dfs().keys() # list keys of underlying dataframes

In [ ]:
card.dfs()['MASS'] # acsess the mass dataframe with standard values 

In this example, we will varry $M_Z$ and $m_{\tau}$ between runs. ParamCard has built-in get and set value functions to more easily acsess and change specific values. To use these functions, we need to know two things: the block in which the value resides (MASS in this case) and the key within that block. If we want to change the tau mass, we need to know MASS and 15, for example. 

In [ ]:
from IPython.display import Math, display

# side trip for fun fact: MadGraph comments show how certain values are calculated
# in particular, the W mass is calculated as a function of Z mass and constants 
W_id = 24
df_masses = card.dfs()['MASS']
df_masses[df_masses['key'] == W_id]['comment'].item() # how is mass for W calculated?

### Set values across two parameters

First, make two lists over which to change the mass values. 

In [ ]:
# want to vary both masses in a range where the physics is highly sensitive -- even if wialdly unphysical
Z_id = 23
tau_id = 15
card = qs.ParamCard(FRAMEWORK_PATH / 'Cards' / 'param_card.dat')
m_z, m_tau = card.get_value('MASS', Z_id), card.get_value('MASS', tau_id) # retrieve standard values
MTAU = np.linspace(1,100,5)*1.777 #vary over a wide range of values
MZ = np.linspace(1,10,5)*91.188   #vary over a wide range of values
print(f"tau masses: {MTAU}\n Z masses (to change W mass indirectly: {MZ}")

Second, generate LHEs for resulting 5x5 pairs of parameter values. Since the LHE reads the current parameter_card.dat, we can: change the values, genorate LHE, repeat.

**Important: MadGraph produces LHEs in no particular order.** We will have to deal with this later when we want to do something useful with the data. 

**Important: when changeing particle masses, the coresponding decay width will not be automatically recalculated unless the entery in the DECAY block is set to -1 explicitly.** I will therefore manually set these values for the W and Z bosons. Noteably, I am neglecting to set the tau decay entery to -1 because it is 0.0 by defult, and so it treated as a final-state particle which does not decay (making it non-zero will break the run). 

In [ ]:
# generate one set of LHE events for every combination of parameter values
card = qs.ParamCard(FRAMEWORK_PATH / 'Cards' / 'param_card.dat')
card.set_value('DECAY', W_id, -1)
card.set_value('DECAY', Z_id, -1)
for mtau in MTAU:
    card.set_value('MASS', tau_id, mtau)
    for mz in MZ:
        print(f'Generating LHE with: mtau = {mtau} mz = {mz}')
        card.set_value('MASS', Z_id, mz)
        qs.generate_LHE(card, FRAMEWORK_PATH)

Third, shower all resulting LHEs. The order is already scrambled, so instead we will reasocate the parameter values with the results by inspecting the return values later. 

In [ ]:
qs.pythia_parallel([tau_id], FRAMEWORK_PATH, 'EXAMPLE_CONTOUR', topics='P_mu', size=1000000)

### Retrieve all files in a dictionary (concat=False)

In this use case, results are independent, each representing a run for a unique set of parameters. The <code>qs.unpickle</code> returns a dictionary of results keyed to run name.

In [ ]:
data = qs.unpickle('EXAMPLE_CONTOUR', concat=False)
data.keys()

### Add $p_{\tau}$ to each set of results:

Each dictionary entry in <code>data</code> contains a tuple (parameters, results) of data type (ParamCard, np.array). For this reason, in the following loop, to acsess the results dataframe, we use v[1] (the right-hand member of the tuple).

In [ ]:
for v in data.values(): # compute pT col in set of results 
    v[1]['pT'] = np.sqrt((v[1]['px']**2)+(v[1]['py']**2)) # v[0] is ParamCard, v[1] is results

### Graph results (in a rational order)

To draw a series of plots in which the order respects the parameter values, we must first ascocate the parameter values with their respective runs, then sort. 

In [ ]:
from scipy import stats
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
fig, axes = plt.subplots(5, 5, figsize=(16, 12))
axes = axes.flatten()
BINS = np.linspace(0, 500, 30)

# Madgraph produces results in arbitrary order
# inspect each parameter setting to re-establish original order
rows = []
for k in data.keys():
    df_masses = data[k][0].dfs()['MASS']
    m_tau = df_masses[df_masses['key'] == tau_id]['value'].item() # setting for mass of tau
    m_z = df_masses[df_masses['key'] == Z_id]['value'].item() # setting for mass of W
    rows.append((k,m_tau,m_z))
df_runs = pd.DataFrame(rows, columns=['run', 'm_tau', 'm_z'])
df_runs.sort_values(['m_tau','m_z'], inplace = True) # original order
ordered_runs = df_runs['run']

for idx, run in enumerate(ordered_runs):
    pc_df = data[run]
    ax = axes[idx]
    ax.hist(pc_df[1]['pT'], bins=BINS)
    m_tau = pc_df[0].get_value('MASS', tau_id)
    m_z = pc_df[0].get_value('MASS', Z_id)
    ax.set_title(f"$m_\\tau$ = {m_tau:.3f},   $m_z$ = {m_z:.3f}")
    ax.set_xlabel('GeV')
    ax.set_yscale('log')

plt.tight_layout()
plt.show()

Note about the results:

These resutls should not be taken to have any sound physical meaning. The range in which the parameters are being veryed is very extreme, and is proably breaking something in some subtle way. The general behivor of the peak shifting to the right with increaced $M_Z$ (and therefor $M_W$) is expected, but I have not checked that the amount of shift is physically consistant. 

This example should be treated as a test of the software, and not a sound physical simulation. 

However, the results would be reliable with more reasonable tweaks to the physics.